<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c34-header.png" alt="Nextia Learning · Anomaly Detection: Find What Is Unusual" width="100%">

# Solution: The feedback loop and its traps

**[The feedback loop and its traps](https://learning.nextia-ai.com/courses/anomaly-detection/m08/the-feedback-loop-and-its-traps/)** · Anomaly Detection: Find What Is Unusual · Module 8, lesson 2 of 3 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** measure the selection bias of reviewed labels, see what it does to a re-ranker, and use a small random sample of reviews to measure what the detector misses.

| | |
|---|---|
| **Time** | About 35 minutes |
| **Needs** | An internet connection for the setup cells (about 7 MB). No account. |
| **You should know** | The re-ranker, from [From unsupervised to semi-supervised](https://learning.nextia-ai.com/courses/anomaly-detection/m08/from-unsupervised-to-semi-supervised/). |
| **You do not need** | A local project. The setup below downloads everything. |
| **Tested** | Python 3.12.13 in Jupyter on macOS (Apple silicon), from a fresh start (*Restart and run all*), with the package versions of the setup cell |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/anomaly-detection/m08/the-feedback-loop-and-its-traps/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C34/notebooks/M08-L02-the-feedback-loop-and-its-traps/feedback-loop-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs the versions that the course uses: pandas 3.1.1, numpy 2.5.4, scipy 1.18.1, scikit-learn 1.9.1, matplotlib 3.11.2. It needs an internet connection. The first time, it takes up to a minute.

You should see one line per package, with the same versions. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again. A restart is needed when an older version was already loaded.

In [ ]:
%pip install -q --disable-pip-version-check pandas==3.1.1 numpy==2.5.4 scipy==1.18.1 scikit-learn==1.9.1 matplotlib==3.11.2
from importlib.metadata import version
for p in ['pandas', 'numpy', 'scipy', 'scikit-learn', 'matplotlib']:
    print(p, version(p))

## Setup: get the course data

Run the next cell. It downloads the data that this lesson uses (`payments.csv.gz`, `sealed-key.txt`) into a folder `anomaly-course/data`, and checks each file's checksum. It also downloads `compare.py` and `features.py`, the course's shared code. It needs an internet connection and no account. It takes about 10 seconds.

You should see `Ready:` and the files with their sizes. If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `anomaly-course` (in Colab: the **Files** panel on the left), then run the cell again.

The data is made up for this course: no real people, cards or payments. [The dataset card](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C34/data/dataset-card.md) describes every table.

In [ ]:
# Setup: download the course data into anomaly-course/data/ and check each file.
# Standard library only.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C34/"
FILES = {
    "payments.csv.gz": "c4e18c1bbaf422cfc99821f2a1b65cf0e8274f835c40ca757a37a531eeccd542",
    "sealed-key.txt": "144cfd16737bdfcc4e992ad8ede0b8856265a2193c89fa7266fe0b27cb62ff98",
}
CODE_FILES = ["compare.py", "features.py"]
HERE = Path.cwd()
# The notebook works in its own folder anomaly-course/, marked with a file, so it never writes into another project.
if Path(".nextia-notebook").exists():          # the cell ran before: start again from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
WORK = Path("anomaly-course").resolve()
(WORK / "data").mkdir(parents=True, exist_ok=True)
(WORK / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

def fetch(name, url, path, expected=None):
    """Use a copy next to the notebook if there is one, else download it."""
    if path.exists() and (expected is None or sha256(path) == expected):
        return
    if (HERE / name).exists():
        shutil.copy(HERE / name, path)
    else:
        download(url, path)
    if expected and sha256(path) != expected:
        raise SystemExit(f"{name} has the wrong checksum. Delete the folder anomaly-course and run again.")

for name, expected in FILES.items():
    fetch(name, LABS + "data/" + name, WORK / "data" / name, expected)
for name in CODE_FILES:
    fetch(name, LABS + name, WORK / name)
os.chdir(WORK)
print("Ready:", ", ".join(f"{n} ({(Path('data') / n).stat().st_size / 1e6:.1f} MB)" for n in FILES))

## Setup: helpers

The next cell defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

It also loads pandas as `pd`, NumPy as `np` and matplotlib as `plt`. Run the cell. It prints nothing.

In [ ]:
# Helpers: check an answer without showing it, and the chart style.
import hashlib, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
pd.set_option("display.precision", 3)
pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 20)
warnings.filterwarnings("ignore", category=FutureWarning)
plt.rcParams.update({"figure.figsize": (9, 3.2), "axes.spines.top": False, "axes.spines.right": False})

def fingerprint(value):
    def norm(v):
        if hasattr(v, "to_dict") and hasattr(v, "columns"):
            return {"columns": [str(c) for c in v.columns], "rows": [norm(list(r)) for r in v.itertuples(index=False)]}
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple, set)):
            return [norm(x) for x in (sorted(v, key=str) if isinstance(v, set) else v)]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 4)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. Candidates, reviews and the answer key

Run the next cell. It builds the candidates and the four weeks of reviews of the last lesson. It also keeps the **kind** of each planted payment (stolen card, card testing, account takeover), so that we can see which kinds get labels.

The course data has an answer key, packed in `sealed-key.txt`. `compare.planted_payments()` unpacks it. You could read it yourself: it is there to score your work, not to keep a secret. A real team has no answer key: it has only the reviews.

> **Check.** You should see 1,400 reviewed alerts, 708 real.

In [ ]:
import compare, features
from sklearn.ensemble import IsolationForest
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
payments = pd.read_csv("data/payments.csv.gz", parse_dates=["ts"])
reference, evaluation = compare.split(payments)
behaviour = features.behaviour(payments)
scaler = StandardScaler().fit(behaviour.loc[reference.index])
forest = IsolationForest(n_estimators=200, random_state=0, n_jobs=-1).fit(scaler.transform(behaviour.loc[reference.index]))
df = evaluation[["payment_id", "ts", "amount"]].copy()
df["day"] = df.ts.dt.floor("D")
df["iso"] = -forest.score_samples(scaler.transform(behaviour.loc[evaluation.index]))
for col in behaviour.columns:
    df[col] = behaviour.loc[evaluation.index, col].to_numpy()
df = df.sort_values(["day", "iso"], ascending=[True, False])
df["rank"] = df.groupby("day").cumcount()
# The answer key: we play the reviewers with it (an assumption: real reviewers make mistakes).
key = compare.planted_payments().set_index("payment_id").kind
df["label"] = df.payment_id.isin(key.index)
df["kind"] = df.payment_id.map(key).fillna("not planted")
candidates = df[df["rank"] < 200].copy()          # each day, the 200 highest scores
candidates["shown"] = candidates["rank"] < 50     # the team reviews the top 50
train_weeks = candidates.day <= "2025-09-28"      # four weeks of reviews
labels = candidates[train_weeks & candidates.shown]
print(f"{len(labels):,} reviewed alerts, {labels.label.sum():,} real")

## 2. The labels are not a fair sample

> **Predict.** 50.6% of the reviewed alerts are real. What share of **all** payments of the four weeks are planted fraud? Write your guess, then run the cell.

In [ ]:
four_weeks = df[df.day <= "2025-09-28"]
unseen = candidates[train_weeks & ~candidates.shown]   # candidates that the team did not see
shares = pd.Series({
    "reviewed alerts (the labels)": labels.label.mean(),
    "all candidates (top 200)": candidates[train_weeks].label.mean(),
    "candidates not shown (51st to 200th)": unseen.label.mean(),
    "all payments": four_weeks.label.mean(),
})
shares.round(3)

> **Check.** You should see 0.506 for the reviewed alerts, 0.157 for all candidates, 0.041 for the candidates not shown and 0.018 for all payments.

The labels are full of fraud **because the detector chose them**. This is **selection bias**: the sample with labels was not chosen at random, so it does not look like the whole.

Now the kinds. Which kinds of fraud get a label, and which ones mostly do not?

In [ ]:
coverage = pd.DataFrame({
    "planted payments": four_weeks[four_weeks.label].kind.value_counts(),
    "with a label": labels[labels.label].kind.value_counts(),
})
coverage["share labelled"] = (coverage["with a label"] / coverage["planted payments"]).round(3)
coverage

> **Check.** Account takeovers: 54 planted payments, 21 with a label.

## 3. What the bias does to the re-ranker

The re-ranker learns mostly from stolen cards and card tests, because those are what the team saw. Run the next cell: it trains the re-ranker of the last lesson and counts the account takeovers in the top 50 of the test weeks.

> **Check.** You should see `iso      precision 0.498, account takeovers found 76` and `re_rank  precision 0.514, account takeovers found 71`.

In [ ]:
FEATS = ["iso"] + list(behaviour.columns)

def train(frame):
    s = StandardScaler().fit(frame[FEATS])
    return s, LogisticRegression(max_iter=2000).fit(s.transform(frame[FEATS]), frame.label)

def top_50(frame, column):
    return frame.sort_values(["day", column], ascending=[True, False]).groupby("day").head(50)

test = candidates[~train_weeks].copy()
s, m = train(labels)
test["re_rank"] = m.predict_proba(s.transform(test[FEATS]))[:, 1]
for column in ("iso", "re_rank"):
    top = top_50(test, column)
    print(f"{column:8} precision {top.label.mean():.3f}, account takeovers found {(top.kind == 'account_takeover').sum()}")

Precision goes up, but the re-ranker finds **fewer** account takeovers than the forest alone. It learned that the alerts it saw most often were real, and account takeovers looked like the "not" alerts.

## 4. A small random sample

The step that reduces selection bias: each day, the team also reviews a few candidates **chosen at random** from those it did not see. Here: 5 a day from the 150 candidates ranked 51st to 200th, for the four weeks. Run the next cell.

> **Check.** You should see `random sample: 140 reviews, 4 real (2.9%); 95% interval 1.1% to 7.1%`.

In [ ]:
rs = np.random.default_rng(11)
sample = unseen.groupby("day", group_keys=False).apply(lambda g: g.sample(5, random_state=int(rs.integers(1 << 30))))
k, n = int(sample.label.sum()), len(sample)
# a 95% interval for a share from a small sample (the Wilson interval)
z = 1.96
centre = (k + z * z / 2) / (n + z * z)
half = z * np.sqrt(k * (n - k) / n + z * z / 4) / (n + z * z)
print(f"random sample: {n} reviews, {k} real ({k / n:.1%}); 95% interval {centre - half:.1%} to {centre + half:.1%}")
print(f"estimate: about {k / n * 150:.1f} real payments a day among the 150 candidates that the team did not see")

The random sample is small, so the interval is wide. But it is an honest estimate: nobody chose these payments because they looked like fraud. The answer key shows the true value. A real team never sees it.

> **Check.** You should see `the answer key: 4.1%, 6.2 a day` and `re-ranker with the random sample: precision 0.514`.

In [ ]:
print(f"the answer key: {unseen.label.mean():.1%}, {unseen.label.sum() / unseen.day.nunique():.1f} a day")
s2, m2 = train(pd.concat([labels, sample]))
test["re_rank_sample"] = m2.predict_proba(s2.transform(test[FEATS]))[:, 1]
print(f"re-ranker with the random sample: precision {top_50(test, 're_rank_sample').label.mean():.3f}")

The random sample does **not** raise precision at the budget here: it added 140 labels with only 4 real ones. Its job is to **measure** what the top 50 misses, and to give labels to payments that the detector did not choose.

## 5. "Not sure" is not "not"

Account takeovers are hard to judge: the customer's own device, the customer's own country. Suppose the team is not sure about them. Run the next cell: it compares writing "not sure" as **not**, and leaving those alerts out as **unknown**.

In [ ]:
# The team is not sure about account takeovers. Two ways to write that down:
written_not = labels.copy()
written_not.loc[written_not.kind == "account_takeover", "label"] = False   # "not sure" written as "not"
left_out = labels[labels.kind != "account_takeover"]                       # "unknown": no label
for name, frame in (("written as not", written_not), ("left out", left_out)):
    s3, m3 = train(frame)
    test[name] = m3.predict_proba(s3.transform(test[FEATS]))[:, 1]
    top = top_50(test, name)
    print(f"{name:15} precision {top.label.mean():.3f}, account takeovers found {(top.kind == 'account_takeover').sum()}")

> **Check.** You should see `written as not  precision 0.509, account takeovers found 59` and `left out        precision 0.510, account takeovers found 61`.

Writing "not sure" as "not" teaches the re-ranker that account takeovers are honest. Leaving them out loses their information, but does not teach the wrong answer. The difference is small here, because only 21 labels are account takeovers. The honest record is three answers: **real**, **not**, and **unknown**.

## 6. Check yourself

**Your turn 1.** What share of the planted account-takeover payments of the four weeks got a label? Write it as `share_ato_labelled`, rounded to 3 decimals (it is in the table `coverage`).

In [ ]:
share_ato_labelled = float(coverage.loc["account_takeover", "share labelled"])

In [ ]:
expect('share_ato_labelled', share_ato_labelled, 0.389, tolerance=0.001)

**Your turn 2.** Take a random sample of **10** candidates a day from `unseen` instead of 5, with `g.sample(10, random_state=1)`. Estimate the real payments a day among the 150 candidates that the team did not see, as above. Write it as `missed_per_day_10`, rounded to 1 decimal.

In [ ]:
sample_10 = unseen.groupby("day", group_keys=False).apply(lambda g: g.sample(10, random_state=1))
missed_per_day_10 = round(float(sample_10.label.mean() * 150), 1)

In [ ]:
expect('missed_per_day_10', missed_per_day_10, 6.4, tolerance=0.001)

In the solution: 6.4 a day with 10 reviews a day; 4.3 with 5; the answer key says 6.2. A larger sample gives a narrower interval, at the cost of more reviews.

## Recap

- Labels exist only where the detector looked: 50.6% of the reviewed alerts are real, against 1.8% of all payments. That is selection bias.
- Kinds that the detector rarely shows rarely get labels: 38.9% of the account takeovers. A re-ranker then learns to push them down.
- A small random sample of reviews measures what the top 50 misses. Record "unknown" honestly, never as "not".

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Explain an alert](https://learning.nextia-ai.com/courses/anomaly-detection/m08/explain-an-alert/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/anomaly-detection/m08/the-feedback-loop-and-its-traps/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The practice data is synthetic and has no real people.